# Patch — Extended regressions with firm / market / sector news indices

Этот блок запускается после обновлённого Notebook 1. Он не удаляет старые результаты, а добавляет новые таблицы:
- `outputs/corr_extended_news_indices.csv`
- `outputs/ols_extended_news_indices.csv`
- `outputs/oos_extended_news_indices.csv`

Основная идея: сравнить старую модель только с `I_t`/`I_firm` и расширенную модель с `I_firm`, `I_market`, `I_sector`.

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

def safe_read_parquet(path: Path) -> pd.DataFrame:
    try:
        return pd.read_parquet(path)
    except Exception:
        return pd.read_parquet(path, engine="fastparquet")

final_path = DATA_DIR / "final_features_daily_extended_contexts.parquet"
if not final_path.exists():
    final_path = DATA_DIR / "final_features_daily.parquet"

df = safe_read_parquet(final_path).copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
df = df.sort_values(["ticker","date"]).reset_index(drop=True)

# Backward compatibility
if "I_firm" not in df.columns and "I_t" in df.columns:
    df["I_firm"] = df["I_t"]

required = ["ticker","date","I_firm","I_market","I_sector"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected extended-news columns: {missing}. Run 01_market_sector_news_patch.ipynb first.")

print(df.shape)
print(df[required].head())


(2512, 32)
  ticker       date    I_firm  I_market  I_sector
0   AAPL 2020-12-29 -0.527073       NaN       NaN
1   AAPL 2020-12-30 -0.122154       NaN       NaN
2   AAPL 2020-12-31       NaN       NaN       NaN
3   AAPL 2021-01-04  0.659435       NaN       NaN
4   AAPL 2021-01-05 -0.162034       NaN       NaN


In [3]:

# --- Ensure target/features exist ---

# Use existing returns if prices are not available.
price_candidates = ["adj_close", "Adj Close", "adjclose", "close", "Close"]
price_col = next((c for c in price_candidates if c in df.columns), None)

if "r_log" not in df.columns:
    if price_col is not None:
        df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
        df["r_log"] = df.groupby("ticker")[price_col].transform(lambda s: np.log(s).diff())
    elif "returns" in df.columns:
        df["returns"] = pd.to_numeric(df["returns"], errors="coerce")
        
        # If returns are simple returns, log-return is log(1 + return).
        # If they are already log returns, this will be very close for small values.
        df["r_log"] = np.log1p(df["returns"])
    else:
        raise ValueError(
            "Cannot find price or returns column. Existing columns: "
            f"{df.columns.tolist()[:30]}"
        )

# Main dependent variable: next-day return.
if "r_log_fwd1" not in df.columns:
    df["r_log_fwd1"] = df.groupby("ticker")["r_log"].shift(-1)

# Changes in technical indicators if available.
for col in ["RSI", "rsi", "MACD", "macd"]:
    if col in df.columns:
        canonical = col.upper() if col.lower() == "rsi" else col.lower()
        dcol = "d_" + canonical
        if dcol not in df.columns:
            df[dcol] = df.groupby("ticker")[col].diff()

# Lags
for col in ["r_log", "I_firm", "I_market", "I_sector"]:
    if col in df.columns:
        for k in range(1, 4):
            df[f"{col}_lag{k}"] = df.groupby("ticker")[col].shift(k)

# Fill count columns with zeros, but leave sentiment indices NaN when no news.
for c in ["n_firm", "n_market", "n_sector", "N_firm_strong", "N_market_strong", "N_sector_strong"]:
    if c in df.columns:
        df[c] = df[c].fillna(0)

safe = df.copy()
print("Prepared:", safe.shape)

preview_cols = [c for c in ["ticker", "date", "returns", "r_log", "r_log_fwd1", "I_firm", "I_market", "I_sector"] if c in safe.columns]
safe[preview_cols].head()


Prepared: (2512, 48)


,ticker,date,returns,r_log,r_log_fwd1,I_firm,I_market,I_sector
0,AAPL,2020-12-29,NaN,NaN,-0.008563,-0.527073,NaN,NaN
1,AAPL,2020-12-30,-0.008527,-0.008563,-0.007732,-0.122154,NaN,NaN
2,AAPL,2020-12-31,-0.007703,-0.007732,-0.025030,NaN,NaN,NaN
3,AAPL,2021-01-04,-0.024719,-0.025030,0.012288,0.659435,NaN,NaN
4,AAPL,2021-01-05,0.012364,0.012288,-0.034241,-0.162034,NaN,NaN


In [4]:

# --- Correlations: target vs firm/market/sector indices ---
corr_rows = []
targets = ["r_log_fwd1"]
for extra_target in ["d_RSI","d_rsi","d_macd","d_MACD"]:
    if extra_target in df.columns:
        targets.append(extra_target)

for ticker, g in df.groupby("ticker"):
    for y in targets:
        for x in ["I_firm","I_market","I_sector"]:
            d = g[[y,x]].dropna()
            corr_rows.append({
                "ticker": ticker,
                "y": y,
                "x": x,
                "n": len(d),
                "corr": d[y].corr(d[x]) if len(d) >= 3 else np.nan,
            })

corr_ext = pd.DataFrame(corr_rows)
corr_ext.to_csv(OUT_DIR / "corr_extended_news_indices.csv", index=False)
display(corr_ext)


,ticker,y,x,n,corr
0,AAPL,r_log_fwd1,I_firm,727,0.005273
1,AAPL,r_log_fwd1,I_market,834,-0.004707
2,AAPL,r_log_fwd1,I_sector,910,-0.059678
3,AAPL,d_RSI,I_firm,726,0.072821
4,AAPL,d_RSI,I_market,834,0.148304
5,AAPL,d_RSI,I_sector,911,-0.006104
6,AAPL,d_macd,I_firm,727,0.079764
7,AAPL,d_macd,I_market,834,0.149923
8,AAPL,d_macd,I_sector,911,0.044095
9,XOM,r_log_fwd1,I_firm,622,0.039453


In [5]:

# --- OLS with HAC / Newey-West SE ---
import statsmodels.api as sm

def fit_ols_hac(data: pd.DataFrame, y: str, xcols: list[str], maxlags: int = 5, model_name: str = ""):
    use = data[[y] + xcols].replace([np.inf, -np.inf], np.nan).dropna()
    if len(use) < max(30, len(xcols) + 10):
        return None
    X = sm.add_constant(use[xcols].astype(float), has_constant="add")
    Y = use[y].astype(float)
    res = sm.OLS(Y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    rows = []
    for p in res.params.index:
        rows.append({
            "model": model_name,
            "y": y,
            "term": p,
            "coef": res.params[p],
            "std_err_HAC": res.bse[p],
            "t": res.tvalues[p],
            "p_value": res.pvalues[p],
            "nobs": int(res.nobs),
            "r2": res.rsquared,
            "aic": res.aic,
            "bic": res.bic,
        })
    return rows

control_candidates = []
for c in ["r_log_lag1", "RSI", "rsi", "MACD", "macd"]:
    if c in df.columns:
        control_candidates.append(c)

specs = {
    "M0_controls_only": control_candidates,
    "M1_firm": ["I_firm"] + control_candidates,
    "M2_firm_market": ["I_firm","I_market"] + control_candidates,
    "M3_firm_market_sector": ["I_firm","I_market","I_sector"] + control_candidates,
}

ols_rows = []
for ticker, g in df.groupby("ticker"):
    for name, xcols in specs.items():
        xcols = [x for x in xcols if x in g.columns]
        rows = fit_ols_hac(g, "r_log_fwd1", xcols, maxlags=5, model_name=name)
        if rows:
            for row in rows:
                row["ticker"] = ticker
            ols_rows.extend(rows)

ols_ext = pd.DataFrame(ols_rows)
ols_ext.to_csv(OUT_DIR / "ols_extended_news_indices.csv", index=False)
display(ols_ext)


,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,M0_controls_only,r_log_fwd1,const,-0.002408,0.002963,-0.812550,0.416476,1253,0.001894,-6576.073173,-6555.539989,AAPL
1,M0_controls_only,r_log_fwd1,r_log_lag1,-0.007850,0.035512,-0.221060,0.825046,1253,0.001894,-6576.073173,-6555.539989,AAPL
2,M0_controls_only,r_log_fwd1,RSI,0.000062,0.000058,1.084776,0.278021,1253,0.001894,-6576.073173,-6555.539989,AAPL
3,M0_controls_only,r_log_fwd1,MACD,-0.000391,0.000252,-1.550069,0.121125,1253,0.001894,-6576.073173,-6555.539989,AAPL
4,M1_firm,r_log_fwd1,const,-0.001650,0.003923,-0.420449,0.674158,725,0.001413,-3860.614948,-3837.684090,AAPL
5,M1_firm,r_log_fwd1,I_firm,0.000143,0.001457,0.098234,0.921746,725,0.001413,-3860.614948,-3837.684090,AAPL
6,M1_firm,r_log_fwd1,r_log_lag1,0.025262,0.056412,0.447819,0.654284,725,0.001413,-3860.614948,-3837.684090,AAPL
7,M1_firm,r_log_fwd1,RSI,0.000028,0.000079,0.359173,0.719465,725,0.001413,-3860.614948,-3837.684090,AAPL
8,M1_firm,r_log_fwd1,MACD,-0.000008,0.000388,-0.019657,0.984317,725,0.001413,-3860.614948,-3837.684090,AAPL
9,M2_firm_market,r_log_fwd1,const,-0.007219,0.005886,-1.226429,0.220037,481,0.003662,-2566.641065,-2541.585861,AAPL


In [6]:

# --- Distributed lag model for firm/market/sector news ---
K = 3
lag_rows = []

for ticker, g in df.groupby("ticker"):
    g = g.copy()
    lag_cols = []
    for base in ["I_firm","I_market","I_sector"]:
        for k in range(0, K+1):
            col = base if k == 0 else f"{base}_lag{k}"
            if col in g.columns:
                lag_cols.append(col)
    xcols = lag_cols + [c for c in ["r_log_lag1","RSI","rsi","MACD","macd"] if c in g.columns]
    rows = fit_ols_hac(g, "r_log_fwd1", xcols, maxlags=5, model_name=f"DL_K{K}_extended")
    if rows:
        for row in rows:
            row["ticker"] = ticker
        lag_rows.extend(rows)

lag_ext = pd.DataFrame(lag_rows)
lag_ext.to_csv(OUT_DIR / "distributed_lag_extended_news_indices.csv", index=False)
display(lag_ext)


,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,DL_K3_extended,r_log_fwd1,const,-0.012211,0.016914,-0.721963,0.470317,48,0.208908,-261.780848,-231.841632,AAPL
1,DL_K3_extended,r_log_fwd1,I_firm,0.001394,0.007202,0.193508,0.846561,48,0.208908,-261.780848,-231.841632,AAPL
2,DL_K3_extended,r_log_fwd1,I_firm_lag1,0.001444,0.005128,0.281581,0.778265,48,0.208908,-261.780848,-231.841632,AAPL
3,DL_K3_extended,r_log_fwd1,I_firm_lag2,-0.002929,0.006192,-0.473013,0.636204,48,0.208908,-261.780848,-231.841632,AAPL
4,DL_K3_extended,r_log_fwd1,I_firm_lag3,-0.005195,0.003623,-1.433842,0.151617,48,0.208908,-261.780848,-231.841632,AAPL
5,DL_K3_extended,r_log_fwd1,I_market,-0.003537,0.002281,-1.550609,0.120995,48,0.208908,-261.780848,-231.841632,AAPL
6,DL_K3_extended,r_log_fwd1,I_market_lag1,0.002790,0.002463,1.132518,0.257417,48,0.208908,-261.780848,-231.841632,AAPL
7,DL_K3_extended,r_log_fwd1,I_market_lag2,-0.003087,0.004523,-0.682366,0.495008,48,0.208908,-261.780848,-231.841632,AAPL
8,DL_K3_extended,r_log_fwd1,I_market_lag3,0.000036,0.003981,0.008988,0.992829,48,0.208908,-261.780848,-231.841632,AAPL
9,DL_K3_extended,r_log_fwd1,I_sector,-0.003735,0.003231,-1.155893,0.247725,48,0.208908,-261.780848,-231.841632,AAPL


In [7]:

# --- Simple out-of-sample comparison: baseline vs firm vs extended ---
from sklearn.metrics import mean_squared_error

oos_rows = []
for ticker, g in df.groupby("ticker"):
    g = g.sort_values("date").copy()
    features_by_model = {
        "baseline_mean": [],
        "M1_firm": ["I_firm"],
        "M2_firm_market": ["I_firm","I_market"],
        "M3_firm_market_sector": ["I_firm","I_market","I_sector"],
    }
    for model_name, feats in features_by_model.items():
        use_cols = ["r_log_fwd1"] + feats
        use = g[use_cols].replace([np.inf,-np.inf], np.nan).dropna()
        if len(use) < 80:
            continue
        split = int(len(use) * 0.7)
        train, test = use.iloc[:split], use.iloc[split:]
        y_train, y_test = train["r_log_fwd1"], test["r_log_fwd1"]
        if model_name == "baseline_mean":
            pred = np.full(len(test), y_train.mean())
        else:
            X_train = sm.add_constant(train[feats].astype(float), has_constant="add")
            X_test = sm.add_constant(test[feats].astype(float), has_constant="add")
            model = sm.OLS(y_train.astype(float), X_train).fit()
            pred = model.predict(X_test)
        oos_rows.append({
            "ticker": ticker,
            "model": model_name,
            "n_train": len(train),
            "n_test": len(test),
            "mse": mean_squared_error(y_test, pred),
        })

oos_ext = pd.DataFrame(oos_rows)
oos_ext.to_csv(OUT_DIR / "oos_extended_news_indices.csv", index=False)
display(oos_ext)


,ticker,model,n_train,n_test,mse
0,AAPL,baseline_mean,878,377,0.000330
1,AAPL,M1_firm,508,219,0.000240
2,AAPL,M2_firm_market,336,145,0.000250
3,AAPL,M3_firm_market_sector,269,116,0.000271
4,XOM,baseline_mean,878,377,0.000201
5,XOM,M1_firm,435,187,0.000211
6,XOM,M2_firm_market,276,119,0.000199
7,XOM,M3_firm_market_sector,231,100,0.000212


## Как читать результаты

- `M1_firm` проверяет старую идею: только новости о компании.
- `M2_firm_market` добавляет общий рыночный фон.
- `M3_firm_market_sector` добавляет также отраслевой/supply-chain фон.

